# modENCODE H3K4me3 L3 Reference Peak Generation (WS190 / ce6 $\rightarrow$ ce11)

Reconstructs the canonical reference peak file `notebooks/reference/modENCODE_H3K4me3_L3_ce11.bed` (4,657 non-overlapping peaks) from raw upstream modENCODE ChIP-seq datasets.

---

### Required Inputs

| Role | Source File / Resource | Description |
|---|---|---|
| **modENCODE L3 Rep 1 Peaks** | `GSM713151_..._macs14_peaks.gff.gz` (GEO GSE28770) | WS190/ce6 MACS 1.4 peak calls (4,408 features, 1-based closed) |
| **modENCODE L3 Rep 2 Peaks** | `GSM713152_..._macs14_peaks.gff.gz` (GEO GSE28770) | WS190/ce6 MACS 1.4 peak calls (3,280 features, 1-based closed) |
| **UCSC Liftover Chain** | `ce6ToCe11.over.chain.gz` (UCSC goldenPath) | Genomic remapping chain from WormBase WS190/ce6 to WBcel235/ce11 |

### Expected Outputs

| Output Role | Target File | Description |
|---|---|---|
| **Canonical Reference Peaks** | `notebooks/reference/modENCODE_H3K4me3_L3_ce11.bed` | 4,657 non-overlapping ce11 genomic intervals (spanning ~7.55% of genome) |

---

### Data Transformation Pipeline: Raw Download to Canonical `ce11` BED

```
NCBI GEO (GSE28770)
  ├── GSM713151: MACS 1.4 peaks (WS190 / ce6, GFF) ────┐
  └── GSM713152: MACS 1.4 peaks (WS190 / ce6, GFF) ────┤
                                                        ▼
                        1. Coordinate & Format Normalization
                           - Add 'chr' prefix to Roman numerals (I -> chrI)
                           - 1-based closed [start, end] -> 0-based half-open [start-1, end)
                           - Output sorted BED files on ce6
                                                        ▼
                        2. Genomic Liftover (WS190/ce6 -> WBcel235/ce11)
                           - UCSC liftOver with ce6ToCe11.over.chain.gz
                           - Remap genomic coordinates to modern assembly
                           - Output lifted BED files for each replicate
                                                        ▼
                        3. Replicate Pooling, Sorting & Merging
                           - Concatenate lifted intervals from both replicates
                           - Sort by canonical chromosome order (chrI..chrX, chrM) and start
                           - Merge overlapping or book-ended intervals (start <= last_end)
                                                        ▼
                        4. Final Canonical Reference Peak File
                           - modENCODE_H3K4me3_L3_ce11.bed (4,657 intervals, ~7.55% coverage)
```

#### 1. Data Ingestion & Upstream Provenance
- **Source**: NCBI GEO [GSE28770](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28770) (modENCODE *C. elegans* H3K4me3 ChIP-seq, N2 L3 stage larvae).
- **Inputs**:
  - `GSM713151_seq-WA30534819_H3K4me3_N2_L3_1_macs14_peaks.gff.gz` (Replicate 1, 4,408 peaks)
  - `GSM713152_seq-WA30534819_H3K4me3_N2_L3_2_macs14_peaks.gff.gz` (Replicate 2, 3,280 peaks)
  - `ce6ToCe11.over.chain.gz` (UCSC liftover chain mapping WormBase WS190/ce6 to WBcel235/ce11).

#### 2. Coordinate & Format Normalization (GFF $\rightarrow$ BED)
The raw modENCODE peak calls are stored in GFF format with:
- **Chromosome identifiers**: Unprefixed Roman numerals (`I`, `II`, `III`, `IV`, `V`, `X`, `M`).
- **Coordinate convention**: 1-based, fully closed intervals `[start, end]`.

**Transformation**:
1. Prefix chromosome names with `chr` (`I` $\rightarrow$ `chrI`) to match UCSC chain file nomenclature.
2. Convert coordinates from 1-based closed to standard 0-based half-open intervals:
   $$\text{bed\_start} = \text{gff\_start} - 1, \quad \text{bed\_end} = \text{gff\_end}$$
3. Sort the resulting intervals by chromosome and start position (`sort -k1,1 -k2,2n`).

#### 3. Genomic Coordinate Liftover (ce6 $\rightarrow$ ce11)
The legacy MACS14 peaks were generated against WormBase WS190 (UCSC `ce6`). To align with modern reference annotations (WBcel235 / UCSC `ce11`):
1. Execute the standalone UCSC `liftOver` binary with `ce6ToCe11.over.chain.gz`.
2. Results per replicate:
   - **GSM713151 (Rep 1)**: 4,407 peaks successfully remapped to ce11 (1 unmapped).
   - **GSM713152 (Rep 2)**: 3,276 peaks successfully remapped to ce11 (4 unmapped).

#### 4. Replicate Pooling, Sorting, and Merging
To construct a single high-confidence, non-redundant reference set:
1. **Concatenate**: Pool all lifted intervals from both replicates (7,683 total lifted intervals).
2. **Sort**: Order intervals canonically by chromosome (`chrI`, `chrII`, `chrIII`, `chrIV`, `chrV`, `chrX`, `chrM`) and ascending start position.
3. **Merge**: Iteratively collapse any overlapping or contiguous (book-ended) intervals across replicates:
   $$\text{If } \text{chrom} = \text{last\_chrom} \text{ and } \text{start} \le \text{last\_end} \implies \text{last\_end} = \max(\text{last\_end}, \text{end})$$
   Otherwise, append as a distinct interval.

#### 5. Output Verification
- Writes the final 3-column BED file to `notebooks/reference/modENCODE_H3K4me3_L3_ce11.bed`.
- Produces exactly **4,657 non-overlapping intervals** spanning 7,571,218 bp (~7.55% of the *C. elegans* genome) with a median width of 1,311 bp.

In [ ]:
import os
import sys
import shutil
import tarfile
import urllib.request
import subprocess
from pathlib import Path
import pandas as pd

NOTEBOOKS_DIR = Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent

REF_DIR = NOTEBOOKS_DIR / "reference"
REF_DIR.mkdir(parents=True, exist_ok=True)

# Remote URLs & Local Targets
CHAIN_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/ce6/liftOver/ce6ToCe11.over.chain.gz"
CHAIN_PATH = REF_DIR / "ce6ToCe11.over.chain.gz"

GSM1_ACCESSION = "GSM713151"
GSM1_FILENAME = "GSM713151_seq-WA30534819_H3K4me3_N2_L3_1_macs14_peaks.gff.gz"
GSM1_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM713nnn/{GSM1_ACCESSION}/suppl/{GSM1_FILENAME}"
GSM1_PATH = REF_DIR / GSM1_FILENAME

GSM2_ACCESSION = "GSM713152"
GSM2_FILENAME = "GSM713152_seq-WA30534819_H3K4me3_N2_L3_2_macs14_peaks.gff.gz"
GSM2_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM713nnn/{GSM2_ACCESSION}/suppl/{GSM2_FILENAME}"
GSM2_PATH = REF_DIR / GSM2_FILENAME

OUTPUT_BED_PATH = REF_DIR / "modENCODE_H3K4me3_L3_ce11.bed"

print(f"Reference directory: {REF_DIR}")
print(f"Target output BED:   {OUTPUT_BED_PATH}")


Reference directory: /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/notebooks/reference
Target output BED:   /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/notebooks/reference/modENCODE_H3K4me3_L3_ce11.bed


### 1. Ensure UCSC `liftOver` is Available

The standalone UCSC `liftOver` binary is required for accurate interval lifting. 
This cell checks `$PATH` and `.venv/bin/`. If missing, it automatically downloads the official UCSC precompiled binary for the host OS directly into `.venv/bin/`.


In [7]:
def ensure_liftover() -> Path:
    """Find liftOver in PATH or install precompiled binary into .venv/bin."""
    liftover_cmd = shutil.which("liftOver")
    if liftover_cmd:
        print(f"Found liftOver in PATH: {liftover_cmd}")
        return Path(liftover_cmd)
    
    venv_bin = REPO_ROOT / ".venv" / "bin"
    venv_liftover = venv_bin / "liftOver"
    if venv_liftover.is_file() and os.access(venv_liftover, os.X_OK):
        print(f"Found liftOver in virtualenv: {venv_liftover}")
        os.environ["PATH"] = f"{venv_bin}:{os.environ.get('PATH', '')}"
        return venv_liftover

    import platform
    system = platform.system()
    machine = platform.machine()
    
    if system == "Darwin":
        arch_dir = "macOSX.arm64" if machine == "arm64" else "macOSX.x86_64"
    elif system == "Linux":
        arch_dir = "linux.x86_64"
    else:
        raise RuntimeError(f"Unsupported OS for automatic liftOver binary download: {system} {machine}")
    
    download_url = f"http://hgdownload.soe.ucsc.edu/admin/exe/{arch_dir}/liftOver"
    dest_dir = venv_bin if venv_bin.exists() else REF_DIR / "bin"
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / "liftOver"
    
    print(f"Downloading precompiled liftOver for {system} ({machine}) from UCSC...")
    urllib.request.urlretrieve(download_url, dest_path)
    dest_path.chmod(0o755)
    os.environ["PATH"] = f"{dest_dir}:{os.environ.get('PATH', '')}"
    print(f"Installed liftOver to: {dest_path}")
    return dest_path

LIFTOVER_EXE = ensure_liftover()
proc = subprocess.run([str(LIFTOVER_EXE)], capture_output=True, text=True)
assert "liftOver - Move annotations" in (proc.stdout + proc.stderr), "Failed to verify liftOver executable"
print("liftOver executable verified successfully.")


Found liftOver in PATH: /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/.venv/bin/liftOver
liftOver executable verified successfully.


### 2. Download Upstream Data & Chain File

- **Chain File**: `ce6ToCe11.over.chain.gz` (~1.4 MB)
- **Peak Sets**: Direct download of the two GFF peak files (~65 KB and ~50 KB) from NCBI GEO FTP. If `GSE28770_RAW.tar` is present in the reference directory, the GFFs will be extracted from it instead.


In [8]:
def download_if_missing(url: str, dest_path: Path, description: str) -> None:
    """Download a remote file if it does not already exist."""
    if dest_path.is_file() and dest_path.stat().st_size > 0:
        print(f"Already cached: {description} ({dest_path.name}, {dest_path.stat().st_size:,} bytes)")
        return
    print(f"Downloading {description} from {url}...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(dest_path, "wb") as out_file:
        shutil.copyfileobj(resp, out_file)
    print(f"Saved: {dest_path.name} ({dest_path.stat().st_size:,} bytes)")

# 1. Download Chain File
download_if_missing(CHAIN_URL, CHAIN_PATH, "UCSC ce6ToCe11 chain file")

# 2. Check for GSE28770_RAW.tar or download per-GSM GFF files
raw_tar = REF_DIR / "GSE28770_RAW.tar"
if raw_tar.is_file() and (not GSM1_PATH.is_file() or not GSM2_PATH.is_file()):
    print(f"Extracting peak files from existing archive: {raw_tar.name}...")
    with tarfile.open(raw_tar) as tar:
        for member in tar.getmembers():
            if member.name in (GSM1_FILENAME, GSM2_FILENAME):
                tar.extract(member, path=REF_DIR)
                print(f"Extracted {member.name}")

download_if_missing(GSM1_URL, GSM1_PATH, "GSM713151 GFF peaks (rep 1)")
download_if_missing(GSM2_URL, GSM2_PATH, "GSM713152 GFF peaks (rep 2)")


Already cached: UCSC ce6ToCe11 chain file (ce6ToCe11.over.chain.gz, 3,755 bytes)
Already cached: GSM713151 GFF peaks (rep 1) (GSM713151_seq-WA30534819_H3K4me3_N2_L3_1_macs14_peaks.gff.gz, 66,125 bytes)
Already cached: GSM713152 GFF peaks (rep 2) (GSM713152_seq-WA30534819_H3K4me3_N2_L3_2_macs14_peaks.gff.gz, 50,743 bytes)


### 3. Convert WS190/ce6 GFF to BED and Execute `liftOver`

modENCODE GFF files use:
- Bare Roman numeral chromosomes (`I`, `II`, ..., `X`, `M`).
- 1-based inclusive start and end coordinates.

We convert these to standard 0-based half-open BED format (`chr` prefix added, `start = gff_start - 1`), sort by coordinate, and pass each replicate to `liftOver`.


In [9]:
import gzip

def lift_gff(gff_gz_path: Path, chain_path: Path, name_prefix: str, liftover_bin: Path) -> Path:
    """Convert GFF WS190 peak file to BED and lift coordinates to ce11."""
    pre_bed = gff_gz_path.parent / f"{name_prefix}.pre.bed"
    srt_bed = gff_gz_path.parent / f"{name_prefix}.ce6.sorted.bed"
    out_bed = gff_gz_path.parent / f"{name_prefix}.ce11.bed"
    unmapped = gff_gz_path.parent / f"{name_prefix}.unmapped.bed"

    # Step 1: Format GFF -> BED
    total_in = 0
    with gzip.open(gff_gz_path, "rt") as in_fh, open(pre_bed, "w") as out_fh:
        for line in in_fh:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 6:
                continue
            total_in += 1
            chrom = f"chr{parts[0]}"
            start = int(parts[3]) - 1  # 1-based inclusive -> 0-based half-open
            end = parts[4]
            score = parts[5]
            out_fh.write(f"{chrom}\t{start}\t{end}\t{name_prefix}_{total_in}\t{score}\n")

    # Step 2: Sort BED
    subprocess.run(f"sort -k1,1 -k2,2n {pre_bed} > {srt_bed}", shell=True, check=True)
    pre_bed.unlink(missing_ok=True)

    # Step 3: Run liftOver
    subprocess.run(
        [str(liftover_bin), str(srt_bed), str(chain_path), str(out_bed), str(unmapped)],
        check=True,
        capture_output=True,
    )
    srt_bed.unlink(missing_ok=True)

    with open(out_bed) as f:
        mapped_count = sum(1 for _ in f)
    with open(unmapped) as f:
        unmapped_count = sum(1 for line in f if not line.startswith("#"))

    print(f"[{name_prefix}] Input peaks: {total_in:,} | Mapped to ce11: {mapped_count:,} | Unmapped: {unmapped_count:,}")
    return out_bed

rep1_ce11 = lift_gff(GSM1_PATH, CHAIN_PATH, "GSM713151_rep1", LIFTOVER_EXE)
rep2_ce11 = lift_gff(GSM2_PATH, CHAIN_PATH, "GSM713152_rep2", LIFTOVER_EXE)


[GSM713151_rep1] Input peaks: 4,408 | Mapped to ce11: 4,407 | Unmapped: 1
[GSM713152_rep2] Input peaks: 3,280 | Mapped to ce11: 3,276 | Unmapped: 4


### 4. Merge Replicates into Canonical Reference BED

The two replicate peak sets are combined, sorted by genomic coordinates, and merged:
- Any overlapping or book-ended intervals are merged into a single contiguous interval.
- Canonical chromosome ordering: `chrI`, `chrII`, `chrIII`, `chrIV`, `chrV`, `chrX`, `chrM`.
- Merging is implemented in pure Python (no external `bedtools` CLI dependency required).


In [10]:
def merge_bed_files(bed_paths: list[Path], output_path: Path) -> int:
    """Concatenate, sort, and merge overlapping genomic intervals."""
    intervals = []
    for p in bed_paths:
        with open(p, "r") as f:
            for line in f:
                if not line.strip():
                    continue
                parts = line.strip().split("\t")
                intervals.append((parts[0], int(parts[1]), int(parts[2])))

    # Canonical chromosome ordering
    chrom_order = {
        "chrI": 1, "chrII": 2, "chrIII": 3,
        "chrIV": 4, "chrV": 5, "chrX": 6, "chrM": 7
    }
    intervals.sort(key=lambda x: (chrom_order.get(x[0], 99), x[0], x[1], x[2]))

    # Merge overlapping or contiguous intervals
    merged = []
    for chrom, start, end in intervals:
        if not merged:
            merged.append([chrom, start, end])
            continue
        last_chrom, last_start, last_end = merged[-1]
        if chrom == last_chrom and start <= last_end:
            merged[-1][2] = max(last_end, end)
        else:
            merged.append([chrom, start, end])

    with open(output_path, "w") as out_f:
        for chrom, start, end in merged:
            out_f.write(f"{chrom}\t{start}\t{end}\n")

    return len(merged)

merged_count = merge_bed_files([rep1_ce11, rep2_ce11], OUTPUT_BED_PATH)
print(f"Successfully generated {OUTPUT_BED_PATH.name} with {merged_count:,} merged intervals.")

# Clean up intermediate replicate bed files
rep1_ce11.unlink(missing_ok=True)
rep2_ce11.unlink(missing_ok=True)
(REF_DIR / "GSM713151_rep1.unmapped.bed").unlink(missing_ok=True)
(REF_DIR / "GSM713152_rep2.unmapped.bed").unlink(missing_ok=True)


Successfully generated modENCODE_H3K4me3_L3_ce11.bed with 4,657 merged intervals.


### 5. Verification & Summary Statistics

We verify:
1. Exact row count equals **4,657**.
2. Peak width distribution and total genome coverage (~7.5% of the ~100.28 Mb *C. elegans* genome).
3. First and last 5 intervals.


In [11]:
df = pd.read_csv(OUTPUT_BED_PATH, sep="\t", header=None, names=["chrom", "start", "end"])
df["width"] = df["end"] - df["start"]

total_bases = df["width"].sum()
CE11_GENOME_SIZE = 100_286_401  # ce11 total nuclear genome bases
genome_pct = (total_bases / CE11_GENOME_SIZE) * 100

print("=" * 60)
print(f"File:               {OUTPUT_BED_PATH}")
print(f"Total Peak Count:   {len(df):,} (Expected: 4,657)")
print(f"Total Bases:        {total_bases:,} bp")
print(f"Genome Coverage:    {genome_pct:.2f}% (Expected: ~7.5%)")
print(f"Median Peak Width:  {df['width'].median():.0f} bp")
print(f"Mean Peak Width:    {df['width'].mean():.1f} bp (min={df['width'].min()}, max={df['width'].max()})")
print("=" * 60)

assert len(df) == 4657, f"Expected 4657 peaks, got {len(df)}"

# Display head and tail
pd.concat([df.head(5), df.tail(5)])


File:               /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/notebooks/reference/modENCODE_H3K4me3_L3_ce11.bed
Total Peak Count:   4,657 (Expected: 4,657)
Total Bases:        7,571,218 bp
Genome Coverage:    7.55% (Expected: ~7.5%)
Median Peak Width:  1311 bp
Mean Peak Width:    1625.8 bp (min=517, max=13779)


,chrom,start,end,width
0,chrI,3764,4561,797
1,chrI,11178,11856,678
2,chrI,16315,17183,868
3,chrI,24047,24723,676
4,chrI,26152,27131,979
4652,chrX,17519602,17520480,878
4653,chrX,17600399,17601581,1182
4654,chrX,17609494,17611446,1952
4655,chrX,17627381,17628510,1129
4656,chrX,17631613,17632917,1304


In [12]:
# Parity Verification against committed baseline
ground_truth_path = REPO_ROOT / "results" / "cut_run092226" / "reference" / "modENCODE_H3K4me3_L3_ce11.bed"

if ground_truth_path.is_file():
    df_gt = pd.read_csv(ground_truth_path, sep="	", header=None, names=["chrom", "start", "end"])
    pd.testing.assert_frame_equal(df[["chrom", "start", "end"]], df_gt[["chrom", "start", "end"]])
    print(f"✓ Exact parity verified: all {len(df):,} intervals match {ground_truth_path.name}")
else:
    print(f"ℹ Ground truth reference file not found at {ground_truth_path} (skipping baseline parity check).")


✓ Exact parity verified: all 4,657 intervals match modENCODE_H3K4me3_L3_ce11.bed


<h3 style="color: red;">⚠️ Ground Truth Reference (Temporary — Remove Once Results Are Trusted)</h3>

> ⚠️ **Warning**: This cell contains legacy ground truth baseline references used for parity verification. Once the pipeline results are fully trusted, this cell and the validation below can be safely removed.

| Role | Source File / Resource | Description |
|---|---|---|
| **Ground Truth Reference** | `results/cut_run092226/reference/modENCODE_H3K4me3_L3_ce11.bed` | Committed reference BED for exact 4,657-interval parity verification |


In [13]:
# Parity Verification against committed baseline
ground_truth_path = REPO_ROOT / "results" / "cut_run092226" / "reference" / "modENCODE_H3K4me3_L3_ce11.bed"

if ground_truth_path.is_file():
    df_gt = pd.read_csv(ground_truth_path, sep="\t", header=None, names=["chrom", "start", "end"])
    pd.testing.assert_frame_equal(df[["chrom", "start", "end"]], df_gt[["chrom", "start", "end"]])
    print(f"✓ Exact parity verified: all {len(df):,} intervals match {ground_truth_path.name}")
else:
    print(f"ℹ Ground truth reference file not found at {ground_truth_path} (skipping baseline parity check).")


✓ Exact parity verified: all 4,657 intervals match modENCODE_H3K4me3_L3_ce11.bed
